# 02 - Toy Word Embeddings with PyTorch

## Learning objective

Train a small feedforward neural network that learns one static embedding vector for every word by predicting a missing middle word.

```text
Context word IDs
-> Embedding lookup
-> Mean pooling
-> Linear output layer
-> Vocabulary logits
-> Cross-entropy loss
-> Automatic backpropagation
-> Parameter update
```

This notebook implements the same concept as the NumPy tutorial, but PyTorch automatically calculates gradients and updates trainable parameters.

No API key, internet connection, LangChain, Gemini, or Tavily access is required.


## 1. Import the required modules

- `re` performs simple tokenization.
- `torch` provides tensors and automatic differentiation.
- `torch.nn` provides neural-network layers and loss functions.
- `torch.nn.functional` provides cosine similarity.


In [1]:
import re

import torch
from torch import nn
from torch.nn import functional as F


torch.manual_seed(42)


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.14.0+cpu
CUDA available: False


## 2. Create a small synthetic corpus

The sentences deliberately repeat related patterns. For example, `buy` and `purchase` appear between similar context words.

Repeating the corpus increases the number of training examples but does not add new linguistic diversity.


In [2]:
sentences = [
    "customers buy products",
    "customers purchase products",
    "people buy goods",
    "people purchase goods",
    "shoppers buy products",
    "shoppers purchase goods",
    "companies sell products",
    "stores sell goods",
    "customers visit stores",
    "shoppers visit stores",
    "dogs chase cats",
    "puppies chase cats",
    "dogs like food",
    "puppies like food",
]


sentences = sentences * 100


print("Number of sentence occurrences:", len(sentences))

for sentence in sentences[:5]:
    print(sentence)


Number of sentence occurrences: 1400
customers buy products
customers purchase products
people buy goods
people purchase goods
shoppers buy products


## 3. Tokenize the corpus

The tokenizer converts text to lowercase and extracts alphabetic words.


In [3]:
def tokenize(text):
    """Convert text to lowercase word tokens."""

    return re.findall(
        pattern=r"[a-z]+",
        string=text.lower(),
    )


tokenized_sentences = [
    tokenize(sentence)
    for sentence in sentences
]


for sentence in tokenized_sentences[:5]:
    print(sentence)


['customers', 'buy', 'products']
['customers', 'purchase', 'products']
['people', 'buy', 'goods']
['people', 'purchase', 'goods']
['shoppers', 'buy', 'products']


## 4. Build the vocabulary and word-ID mappings

Every distinct word receives an integer ID. The IDs are only lookup positions and contain no semantic meaning.


In [4]:
vocabulary = sorted({
    word
    for sentence in tokenized_sentences
    for word in sentence
})


vocabulary_size = len(vocabulary)


word_to_id = {
    word: index
    for index, word in enumerate(vocabulary)
}


id_to_word = {
    index: word
    for word, index in word_to_id.items()
}


print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", vocabulary)


for word, word_id in word_to_id.items():
    print(f"{word:10s} -> {word_id}")


Vocabulary size: 17
Vocabulary: ['buy', 'cats', 'chase', 'companies', 'customers', 'dogs', 'food', 'goods', 'like', 'people', 'products', 'puppies', 'purchase', 'sell', 'shoppers', 'stores', 'visit']
buy        -> 0
cats       -> 1
chase      -> 2
companies  -> 3
customers  -> 4
dogs       -> 5
food       -> 6
goods      -> 7
like       -> 8
people     -> 9
products   -> 10
puppies    -> 11
purchase   -> 12
sell       -> 13
shoppers   -> 14
stores     -> 15
visit      -> 16


## 5. Create missing-word training examples

For the sentence:

```text
customers buy products
```

The model receives:

```text
Context: customers, products
Target: buy
```

The first and third words become the input. The middle word becomes the label.


In [6]:
context_examples = []
target_examples = []


for sentence in tokenized_sentences:
    left_word, target_word, right_word = sentence

    context_examples.append([
        word_to_id[left_word],
        word_to_id[right_word],
    ])

    target_examples.append(
        word_to_id[target_word]
    )


context_tensor = torch.tensor(
    context_examples,
    dtype=torch.long,
)


target_tensor = torch.tensor(
    target_examples,
    dtype=torch.long,
)


print("Context shape:", context_tensor.shape)
print("Target shape:", target_tensor.shape)


print("First eight examples:")

for context_ids, target_id in zip(
    context_tensor[:8],
    target_tensor[:8],
):
    context_words = [
        id_to_word[word_id.item()]
        for word_id in context_ids
    ]

    target_word = id_to_word[target_id.item()]

    print(
        "Context:", context_words,
        "Target:", target_word,
    )


Context shape: torch.Size([1400, 2])
Target shape: torch.Size([1400])
First eight examples:
Context: ['customers', 'products'] Target: buy
Context: ['customers', 'products'] Target: purchase
Context: ['people', 'goods'] Target: buy
Context: ['people', 'goods'] Target: purchase
Context: ['shoppers', 'products'] Target: buy
Context: ['shoppers', 'goods'] Target: purchase
Context: ['companies', 'products'] Target: sell
Context: ['stores', 'goods'] Target: sell


## 6. Define the feedforward neural network

The network contains two trainable layers:

1. `nn.Embedding` stores one trainable vector per vocabulary word.
2. `nn.Linear` converts the pooled context vector into one score for every vocabulary word.

Mean pooling contains no trainable parameters.


In [7]:
class MissingWordModel(nn.Module):
    """Predict a missing middle word from two context words."""

    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocabulary_size,
            embedding_dim=embedding_dimension,
        )

        self.output_layer = nn.Linear(
            in_features=embedding_dimension,
            out_features=vocabulary_size,
        )

    def forward(self, context_word_ids):
        """Return one vocabulary logit per training example."""

        context_embeddings = self.embedding(
            context_word_ids
        )

        context_vector = context_embeddings.mean(
            dim=1
        )

        logits = self.output_layer(
            context_vector
        )

        return logits


## 7. Create and inspect the model

With 17 words and five values per word:

```text
Embedding weight shape: 17 x 5
Linear weight shape:    17 x 5
Linear bias shape:      17
```

PyTorch stores a linear layer's weights as:

```text
out_features x in_features
```

The layer still performs the conceptual transformation:

```text
5 context features -> 17 vocabulary scores
```


In [9]:
embedding_dimension = 5


model = MissingWordModel(
    vocabulary_size=vocabulary_size,
    embedding_dimension=embedding_dimension,
)


print(model)


print("Trainable parameters:")

for parameter_name, parameter in model.named_parameters():
    print(
        f"{parameter_name:22s}",
        tuple(parameter.shape),
        "parameters =",
        parameter.numel(),
    )


number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


print("Total trainable parameters:", number_of_parameters)


MissingWordModel(
  (embedding): Embedding(17, 5)
  (output_layer): Linear(in_features=5, out_features=17, bias=True)
)
Trainable parameters:
embedding.weight       (17, 5) parameters = 85
output_layer.weight    (17, 5) parameters = 85
output_layer.bias      (17,) parameters = 17
Total trainable parameters: 187


## 8. Inspect the initial embedding matrix

`model.embedding.weight` is the trainable embedding matrix.

Every row corresponds to one vocabulary word. At initialization, the values are random.


In [11]:
print("Embedding matrix shape:", model.embedding.weight.shape)


customers_id = word_to_id["customers"]


print("Initial embedding for 'customers':")
print(model.embedding.weight[customers_id].detach())


Embedding matrix shape: torch.Size([17, 5])
Initial embedding for 'customers':
tensor([-0.7581,  1.0783,  0.8008,  1.6806,  1.2791])


## 9. Inspect one forward pass before training

The first dimension is the batch size. The model returns one logit for every possible target word.


In [12]:
example_context = context_tensor[:1]


with torch.no_grad():
    example_embeddings = model.embedding(
        example_context
    )

    example_context_vector = example_embeddings.mean(
        dim=1
    )

    example_logits = model(
        example_context
    )

    example_probabilities = torch.softmax(
        example_logits,
        dim=1,
    )


print("Input IDs:", example_context)
print("Input words:", [
    id_to_word[word_id.item()]
    for word_id in example_context[0]
])
print("Context embeddings shape:", example_embeddings.shape)
print("Context vector shape:", example_context_vector.shape)
print("Logits shape:", example_logits.shape)
print("Probability sum:", example_probabilities.sum().item())


highest_id = example_probabilities.argmax(dim=1).item()


print("Initial highest-probability word:", id_to_word[highest_id])
print("Correct target word:", id_to_word[target_tensor[0].item()])


Input IDs: tensor([[ 4, 10]])
Input words: ['customers', 'products']
Context embeddings shape: torch.Size([1, 2, 5])
Context vector shape: torch.Size([1, 5])
Logits shape: torch.Size([1, 17])
Probability sum: 0.9999998807907104
Initial highest-probability word: visit
Correct target word: buy


## 10. Define the loss function and optimizer

`CrossEntropyLoss` combines the softmax and negative log-likelihood calculations required for multiclass prediction.

`Adam` updates every trainable parameter returned by `model.parameters()`:

- `embedding.weight`
- `output_layer.weight`
- `output_layer.bias`


In [13]:
loss_function = nn.CrossEntropyLoss()


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05,
)


print("Loss function:", loss_function)
print("Optimizer:", optimizer.__class__.__name__)


Loss function: CrossEntropyLoss()
Optimizer: Adam


## 11. Train the model

The four critical lines are:

```python
optimizer.zero_grad()
logits = model(context_tensor)
loss.backward()
optimizer.step()
```

PyTorch's automatic differentiation calculates the gradients that were written manually in the NumPy tutorial.


In [14]:
number_of_epochs = 500
loss_history = []


for epoch in range(number_of_epochs):

    optimizer.zero_grad()

    logits = model(
        context_tensor
    )

    loss = loss_function(
        logits,
        target_tensor,
    )

    loss.backward()

    optimizer.step()

    loss_history.append(
        loss.item()
    )

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:3d}"
            f" | Loss: {loss.item():.4f}"
        )


print(
    f"Final loss after {number_of_epochs} epochs:"
    f" {loss_history[-1]:.4f}"
)


Epoch   0 | Loss: 2.5877
Epoch  50 | Loss: 0.2768
Epoch 100 | Loss: 0.2043
Epoch 150 | Loss: 0.2005
Epoch 200 | Loss: 0.1995
Epoch 250 | Loss: 0.1990
Epoch 300 | Loss: 0.1987
Epoch 350 | Loss: 0.1986
Epoch 400 | Loss: 0.1984
Epoch 450 | Loss: 0.1984
Final loss after 500 epochs: 0.1983


## 12. Inspect the training loss

The loss should generally decline as the network gets better at predicting the missing target words.


In [15]:
print("Initial loss:", round(loss_history[0], 4))
print("Final loss:", round(loss_history[-1], 4))


for epoch in [0, 50, 100, 200, 300, 400, 499]:
    print(
        f"Epoch {epoch:3d}:"
        f" {loss_history[epoch]:.4f}"
    )


Initial loss: 2.5877
Final loss: 0.1983
Epoch   0: 2.5877
Epoch  50: 0.2768
Epoch 100: 0.2043
Epoch 200: 0.1995
Epoch 300: 0.1987
Epoch 400: 0.1984
Epoch 499: 0.1983


## 13. Inspect gradients

PyTorch stores the most recently calculated gradient in each parameter's `.grad` attribute.

After `optimizer.step()`, the parameters have already been updated, but the gradient tensors remain available until `optimizer.zero_grad()` is called again.


In [16]:
for parameter_name, parameter in model.named_parameters():
    print(
        f"{parameter_name:22s}",
        "gradient shape =",
        tuple(parameter.grad.shape),
        "gradient norm =",
        round(parameter.grad.norm().item(), 6),
    )


embedding.weight       gradient shape = (17, 5) gradient norm = 0.000181
output_layer.weight    gradient shape = (17, 5) gradient norm = 0.00026
output_layer.bias      gradient shape = (17,) gradient norm = 1.9e-05


## 14. Retrieve learned word embeddings

The final word embeddings are rows of `model.embedding.weight`.

`torch.no_grad()` indicates that these inspection operations do not need gradient tracking.


In [17]:
def get_embedding(word):
    """Return a detached copy of one learned word embedding."""

    word_id = word_to_id[word]

    with torch.no_grad():
        embedding = model.embedding.weight[
            word_id
        ].detach().clone()

    return embedding


for word in [
    "customers",
    "people",
    "shoppers",
    "products",
    "goods",
    "dogs",
    "puppies",
]:
    print(
        f"{word:10s}",
        get_embedding(word),
    )


customers  tensor([ 1.6606,  0.3809, -0.0595,  3.4995,  2.0021])
people     tensor([ 1.4122, -4.9341, -2.1209,  0.5549, -3.4897])
shoppers   tensor([ 2.4703, -2.9884, -0.1883,  1.8976, -0.5906])
products   tensor([ 0.1425, -0.6856, -2.3903,  2.7396, -3.2162])
goods      tensor([ 5.3685,  3.8755, -2.5322,  0.1857, -1.5249])
dogs       tensor([-0.8481,  0.3912, -0.1339, -1.9436,  1.1807])
puppies    tensor([-2.0281, -0.2346, -1.1047, -1.2782,  1.2365])


## 15. Compare words using cosine similarity

Cosine similarity compares the directions of two vectors.

The exact values are not important in this tiny synthetic example. Focus on relative similarities.


In [18]:
def compare_words(first_word, second_word):
    """Print cosine similarity between two word embeddings."""

    similarity = F.cosine_similarity(
        get_embedding(first_word).unsqueeze(0),
        get_embedding(second_word).unsqueeze(0),
    )

    print(
        f"{first_word:10s}"
        f" versus "
        f"{second_word:10s}"
        f" = {similarity.item():.3f}"
    )


compare_words("customers", "shoppers")
compare_words("customers", "people")
compare_words("products", "goods")
compare_words("dogs", "puppies")
compare_words("customers", "dogs")


customers  versus shoppers   = 0.442
customers  versus people     = -0.155
products   versus goods      = 0.269
dogs       versus puppies    = 0.795
customers  versus dogs       = -0.528


## 16. Find the nearest words

This compares the query word with every other row of the learned embedding matrix.


In [20]:
def find_similar_words(
    query_word,
    number_of_results=5,
):
    """Return words nearest to the query embedding."""

    query_embedding = get_embedding(
        query_word
    )

    similarities = []

    for candidate_word in vocabulary:

        if candidate_word == query_word:
            continue

        similarity = F.cosine_similarity(
            query_embedding.unsqueeze(0),
            get_embedding(candidate_word).unsqueeze(0),
        ).item()

        similarities.append(
            (candidate_word, similarity)
        )

    similarities.sort(
        key=lambda item: item[1],
        reverse=True,
    )

    return similarities[:number_of_results]


for query_word in [
    "customers",
    "products",
    "dogs",
]:
    print(
        f"Words similar to '{query_word}':"
    )

    for word, similarity in find_similar_words(
        query_word
    ):
        print(
            f"  {word:10s}"
            f" {similarity:.3f}"
        )


Words similar to 'customers':
  purchase   0.642
  stores     0.550
  shoppers   0.442
  sell       0.409
  visit      0.359
Words similar to 'products':
  people     0.663
  shoppers   0.465
  chase      0.388
  like       0.299
  goods      0.269
Words similar to 'dogs':
  puppies    0.795
  food       0.556
  cats       0.519
  buy        0.443
  companies  0.417


## 17. Predict a missing middle word

The prediction function performs a forward pass and converts logits to probabilities.

`torch.topk` retrieves the predictions with the largest probabilities.


In [21]:
def predict_missing_word(
    left_word,
    right_word,
    number_of_results=5,
):
    """Predict the missing middle word."""

    context_ids = torch.tensor(
        [[
            word_to_id[left_word],
            word_to_id[right_word],
        ]],
        dtype=torch.long,
    )

    with torch.no_grad():
        logits = model(context_ids)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        top_probabilities, top_ids = torch.topk(
            probabilities,
            k=number_of_results,
            dim=1,
        )

    predictions = []

    for word_id, probability in zip(
        top_ids[0],
        top_probabilities[0],
    ):
        predictions.append((
            id_to_word[word_id.item()],
            probability.item(),
        ))

    return predictions


predictions = predict_missing_word(
    left_word="customers",
    right_word="products",
)


print("customers [MASK] products")

for word, probability in predictions:
    print(
        f"  {word:10s}"
        f" probability = {probability:.3f}"
    )


customers [MASK] products
  buy        probability = 0.500
  purchase   probability = 0.500
  visit      probability = 0.000
  sell       probability = 0.000
  cats       probability = 0.000


## 18. Try additional predictions


In [23]:
test_contexts = [
    ("people", "goods"),
    ("companies", "products"),
    ("dogs", "cats"),
    ("puppies", "food"),
]


for left_word, right_word in test_contexts:
    print(
        f"{left_word} [MASK] {right_word}"
    )

    for word, probability in predict_missing_word(
        left_word,
        right_word,
        number_of_results=3,
    ):
        print(
            f"  {word:10s}"
            f" {probability:.3f}"
        )


people [MASK] goods
  purchase   0.500
  buy        0.500
  sell       0.000
companies [MASK] products
  sell       1.000
  like       0.000
  purchase   0.000
dogs [MASK] cats
  chase      1.000
  like       0.000
  purchase   0.000
puppies [MASK] food
  like       1.000
  chase      0.000
  sell       0.000


## 19. Verify which parameters are trainable

A parameter is trainable when `requires_grad` is `True` and the optimizer receives it through `model.parameters()`.


In [24]:
for parameter_name, parameter in model.named_parameters():
    print(
        f"{parameter_name:22s}",
        "requires_grad =",
        parameter.requires_grad,
    )


embedding.weight       requires_grad = True
output_layer.weight    requires_grad = True
output_layer.bias      requires_grad = True


## 20. NumPy versus PyTorch

### NumPy implementation

The tutorial manually calculated:

- Gradient of vocabulary logits
- Gradient of output weights
- Gradient of output bias
- Gradient of context vectors
- Gradient of embedding rows
- Gradient-descent updates

### PyTorch implementation

The same work is performed by:

```python
loss.backward()
optimizer.step()
```

The network architecture remains explicit:

```text
nn.Embedding
-> mean(dim=1)
-> nn.Linear
-> CrossEntropyLoss
```

## Central takeaway

Both the embedding layer and output layer contain trainable parameters. The missing-word prediction loss supplies the learning signal that updates both layers.


## 21. Important limitations

- The corpus is tiny and synthetic.
- Repeated sentences do not add linguistic variety.
- Mean pooling loses word order.
- Each vocabulary word receives one static vector.
- The same word cannot have different vectors in different contexts.
- Modern transformer embedding models use contextual token representations and much larger training corpora.

## Practice exercises

1. Change the embedding dimension from 5 to 2, 10, or 20.
2. Change the optimizer from Adam to SGD.
3. Change the learning rate.
4. Add more sentence patterns.
5. Remove the corpus repetition and compare the loss.
6. Freeze the embedding layer and observe what changes.
7. Tie the output weights to the embedding matrix.
8. Rebuild the model as a Skip-gram objective.
